# 05 · Inspect movement preservation on GAVD video

**Does the behavior survive real images?** Use the existing GAVD manifest
and videos to inspect visible changes and uncertainty. This notebook
does not treat a pose estimator or motion prior as clinical reference truth.

Run each notebook in a fresh kernel, in order **00 → 04**. Notebook 05 is
an external visual stress test. These are offline restoration experiments:
the model may inspect the declared complete clip. They do not claim causal
forecasting or clinical diagnosis.

**The default is real data.** Export `MP_RUN_ROOT`, the AMASS and GAVD data
paths, and the model configuration before opening Jupyter. See the
[launch guide](../../slurm/motion-preservation/README.md).
For a CPU walkthrough of the mechanics, explicitly choose `MP_MODE=demo`
and a separate run directory. Demo outputs cannot establish a research result.

[Proposal](../../docs/studies/motion-preservation/protocol/proposal.md)
· [Notebook guide](README.md)

In [ ]:
from pathlib import Path
import json
import os
import sys
from time import perf_counter

project_override = os.environ.get("GAVD6_ROOT")
candidates = ([Path(project_override).expanduser()] if project_override else
              [Path.cwd(), *Path.cwd().parents])
PROJECT_ROOT = next((p.resolve() for p in candidates
                     if (p / "src/gavd6_sjepa").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Set GAVD6_ROOT to the gavd6 checkout.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)  # Resolve manifest/config paths from the checkout in every kernel.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, Markdown, Video, display
from gavd6_sjepa.research_directions.motion_preservation import workflow, plots

get_ipython().run_line_magic("matplotlib", "inline")
plt.rcParams.update({"figure.figsize": (9, 3.5), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})
cfg = workflow.config_from_environment()
RUN_ROOT = Path(cfg.run_root)
print(f"Mode: {cfg.mode}; run directory: {RUN_ROOT}")
if cfg.mode == "demo":
    display(Markdown("**DEMO ONLY: generated fixtures and stand-in models. "
                     "These outputs are not evidence about AMASS, GAVD, or a pretrained prior.**"))

## 1. Select clips before comparing model errors

GAVD provides source videos, annotated sequence boundaries and descriptive
gait labels. Use the manifest to choose a small fixed stress set with
clear motion, occlusion and poor tracking. Keep a source recording in
one role. Do not infer participant identity from recording identity.

The experiment uses full declared clips for offline restoration. It is
not forecasting. The existing GAVD confirmation cohort must remain
excluded. Set `MP_GAVD_RESERVATION` to that study's existing
`config/source-reservation.csv`; without it, the stress stage reports
that it has not run and does not decode a potentially reserved clip.

In [ ]:
inventory = workflow.inventory(cfg)
gavd_manifest = inventory["gavd"]
useful_columns = [c for c in ("sequence_id", "video_id", "first_frame", "last_frame",
                               "cam_view", "gait_pattern_annotation", "available") if c in gavd_manifest]
display(gavd_manifest[useful_columns].head(15) if useful_columns else gavd_manifest.head(15))

## 2. Run the configured external stress path

The implemented first pass estimates flow on real RGB frames. It can
compare supplied observed and repaired trajectories in full-image
coordinates. This is a measurement stress test; it does not automatically
apply the trained 3D gate to video. A body-model reconstruction such as
WHAM is an estimate, not metric 3D truth.

Optional pose exports contain `source_frames` (zero-based original video
frame IDs), `joints2d[N,22,2]` in full-frame pixels and
`coordinate_system="full_frame_pixels"`. Add `repaired_joints2d` when
available. A 3D WHAM output must first be projected with its actual camera.
It cannot be interpreted as 2D pixels by relabelling the coordinates.

In [ ]:
started = perf_counter()
stress = workflow.gavd_stress(cfg)
print(f"GAVD stress stage completed in {perf_counter() - started:.1f} seconds.")
display(stress)

## 3. Watch the evidence alongside the repaired motion

For each saved overlay, examine the original image, observed path, prior
path and final path together. Useful questions are whether the visible
motion survives, whether a clear tracking slip is reduced, and whether
the gate reports uncertainty when neither estimate is supported.

The viewer below embeds the first available saved video. If the stage
produced a table of external paths instead, open those paths on HAIC.

In [ ]:
video_columns = [c for c in stress if any(word in c.lower() for word in ("overlay", "video_path", "preview", "gallery"))]
displayed_video = False
for column in video_columns:
    for value in stress[column].dropna().head(3):
        path = Path(str(value)).expanduser()
        if not path.is_absolute():
            path = RUN_ROOT / path
        if path.is_file() and path.suffix.lower() in {".mp4", ".webm"}:
            display(Video(str(path), embed=True, width=720))
            displayed_video = True
            break
    if displayed_video:
        break
if not displayed_video:
    print("No local video preview was returned. Inspect the stress table and stage outputs.")

## 4. Record observations without inventing ground truth

A useful review records visible evidence, an unresolved ambiguity and
the affected time range. Avoid labels such as “clinically corrected.”
Lower model disagreement or smoother motion does not establish that a
person's real movement was preserved.

Flow agreement, missingness and abstention can be measured directly as
diagnostics. They do not replace known 3D reference motion. Binary
normal-versus-abnormal classification is not the headline and is not
required by this notebook.

In [ ]:
# A blank review table for manual observations; no judgments are inferred.
id_column = next((c for c in ("sequence_id", "case_id", "video_id") if c in stress), None)
identifiers = stress[id_column].astype(str).tolist() if id_column else []
review = pd.DataFrame({"clip": identifiers, "visible_evidence": "",
                       "time_range": "", "unresolved_ambiguity": ""})
display(review)
# After writing observations, save explicitly:
# review.to_csv(RUN_ROOT / "gavd_manual_observations.csv", index=False)

## Final interpretation

A strong result combines a passing controlled preservation-versus-repair
test, transfer across reserved events, and credible behavior on natural
motion. A synthetic gain with visible failures on real video is a
limitation to understand before expanding the claim.

Return to the [notebook guide](README.md) for the execution sequence and
the distinction between implemented experiments and external comparisons.